# Independent raw-event analysis

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import csv,json,pickle,hashlib,sys,math
import numpy as np
from contract import ROOT as REPO,REFERENCE,CODE,PROV,read_json,write_json,sha,verify_inputs
PHASE=REFERENCE
ROOT=REPO/'outputs/analysis'
ROOT.mkdir(parents=True,exist_ok=True)
def csv_out(path,rows):
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def csvread(path):
    with Path(path).open() as f:return list(csv.DictReader(f))

def analyze():
    parameters=[];folder=CODE/'affectively/models/solid/classifier/Linear'
    for i in range(5):
        with (folder/f'Cluster_0_classifier_preferences_linear_{i}.pkl').open('rb') as f:m=pickle.load(f)
        with (folder/f'Cluster_0_classifier_preferences_linear_scaler_{i}.pkl').open('rb') as f:s=pickle.load(f)
        assert list(m.classes_)==[0,1];parameters.append((m.coef_[0],float(m.intercept_[0]),s.scale_,s.min_))
    def q_calc(pair):return float(np.mean([1/(1+np.exp(np.dot(np.clip(pair*scale+offset,0,1),coef)+bias)) for coef,bias,scale,offset in parameters]))
    episodes=[];trajectories={};pair_rows=[];segments=[];states=set();pids=set();random_builds={};errors={'q':0.,'reward':0.,'pair':0.};all_decisions=0
    for condition in ['TASK','MAX','RANDOM']:
        attempt=PHASE/'runs'/('P1-'+condition)/'attempt-01';cfg=read_json(attempt/'resolved_config.json')
        assert all(sha(CODE/k.removeprefix('code/'))==v for k,v in cfg['input_hashes'].items() if k.startswith('code/'))
        for seed in range(3001,3011):
            d=attempt/f'evaluate-{seed}';result=read_json(d/'result.json');supervisor=read_json(attempt/f'evaluate-{seed}-supervision.json')
            assert result['status']=='PASS' and supervisor['exit_code']==0 and not supervisor['timed_out']
            assert not supervisor['unity_alive_after_worker'] and supervisor['no_tcp_connection_after'] and supervisor['port_reusable']
            rt=result['runtime'];assert rt['unity_exited'] and rt['static_inputs_preserved'] and not rt['errors']
            assert rt['initialization_handshake_seeds']==[seed] and rt['estimator_state_before']==rt['estimator_state_after']
            states.add(rt['estimator_state_after']);pids.add(result['pid'])
            assert result['config_sha256']==sha(attempt/'resolved_config.json') and result['PPO_learning_decisions']==0
            rows=[json.loads(line) for line in (d/'events.jsonl').read_text().splitlines()]
            resets=[r for r in rows if r['event']=='reset'];rows=[r for r in rows if r['event']=='decision'];assert len(resets)==1
            np.testing.assert_array_equal(resets[0]['observation'][-5:],0)
            assert len(rows)==result['evaluation_decisions'] and 0<len(rows)<=600
            rng=np.random.default_rng(seed+1000);features=[];previous=None;q=None;last=None;score=0.;fresh_q=[];seen=set()
            for decision,row in enumerate(rows,1):
                all_decisions+=1;assert decision==row['decision']==row['global_decision']
                if condition=='RANDOM':
                    assert result['action_sampler_seed']==seed+1000
                    np.testing.assert_array_equal(row['action'],rng.integers(0,3,size=2,dtype=np.int64))
                assert row['native_action']==row['action']+[0] and all(0<=a<3 for a in row['action'])
                raw=np.asarray(row['game_observation'],dtype=float);assert raw.shape==(81,) and np.isfinite(raw).all()
                b=float(row['raw_score_signal']>score);score=max(score,row['raw_score_signal']);features.append(raw[-27:]);fresh=False
                if decision%15==0:
                    mean=np.mean(features[-15:],axis=0)
                    if previous is not None:
                        pair=np.r_[previous,mean];q=q_calc(pair);fresh=True;last=decision;fresh_q.append(q)
                        pair_id=tuple(row['pair_id']);assert pair_id==(1,decision//15-1,decision//15) and pair_id not in seen;seen.add(pair_id)
                        np.testing.assert_allclose(row['pair_input'],pair,atol=1e-6,rtol=1e-5)
                        errors['pair']=max(errors['pair'],float(np.max(np.abs(np.asarray(row['pair_input'])-pair))))
                        pair_rows.append({'condition':condition,'requested_seed':seed,'decision':decision,'game_seconds':decision*.2,'q':q,
                            'raw_score':row['raw_score_signal'],'task_b':b,'clipped_member_features':row['estimator']['clipped_member_features']})
                    previous=mean
                assert row['fresh']==fresh and row['valid']==(q is not None)
                if q is None:assert row['q'] is None
                else:
                    np.testing.assert_allclose(row['q'],q,atol=1e-6,rtol=1e-5);errors['q']=max(errors['q'],abs(row['q']-q))
                affect=q if fresh else 0.;reward=affect if condition=='MAX' else b
                np.testing.assert_allclose([row['b'],row['delivered_reward'],row['affect_impulse']],[b,reward,affect],atol=1e-6,rtol=1e-5)
                errors['reward']=max(errors['reward'],abs(row['delivered_reward']-reward))
                age=0 if last is None else decision-last
                expected=np.r_[raw,[decision/600,0 if q is None else q,float(q is not None),float(fresh),age/600]].astype(np.float32)
                np.testing.assert_allclose(row['observation'],expected,atol=1e-6,rtol=1e-5)
            last_row=rows[-1];assert last_row['terminated'] or last_row['truncated']
            summary=read_json(d/'episode_summaries.json');assert len(summary)==1 and summary[0]['decisions']==len(rows)
            np.testing.assert_allclose(summary[0]['task_return'],sum(r['b'] for r in rows))
            np.testing.assert_allclose(summary[0]['delivered_return'],sum(r['delivered_reward'] for r in rows))
            compact=[{k:r[k] for k in ['decision','derived_game_seconds','raw_score_signal','score_high_water','b','q','valid','fresh','affect_impulse','delivered_reward','terminated','truncated']} for r in rows]
            trajectories[condition+'-'+str(seed)]=compact
            trajectory_hash=hashlib.sha256(json.dumps([{k:r[k] for k in ['action','game_observation','q','raw_score_signal']} for r in rows],sort_keys=True).encode()).hexdigest()
            eligible=max(len(rows)//15-1,0)
            ep={'condition':condition,'requested_Unity_seed':seed,'action_seed':seed+1000 if condition=='RANDOM' else None,
                'training_decisions':0 if condition=='RANDOM' else 51200,'decisions':len(rows),'raw_score':last_row['raw_score_signal'],
                'score_high_water':score,'task_events':sum(r['b'] for r in rows),'delivered_return':sum(r['delivered_reward'] for r in rows),
                'zero_progress':score<=0 and sum(r['b'] for r in rows)==0,'q_mean':float(np.mean(fresh_q)),'q_min':min(fresh_q),'q_max':max(fresh_q),
                'valid_pairs':len(fresh_q),'eligible_pairs':eligible,'pair_coverage':len(fresh_q)/eligible if eligible else None,'horizon_pair_coverage':len(fresh_q)/39,
                'invalid_warmup_decisions':sum(not r['valid'] for r in rows),'missing_eligible_pairs':eligible-len(fresh_q),
                'success_completion':None,'terminated':last_row['terminated'],'truncated':last_row['truncated'],'end_reason':last_row['episode_end_reason'],
                'worker_wall_seconds':result['wall_seconds'],'observed_loop_seconds':rows[-1]['wall_seconds'],
                'nonloop_overhead_seconds':result['wall_seconds']-rows[-1]['wall_seconds'],'trajectory_sha256':trajectory_hash,
                'config_sha256':sha(attempt/'resolved_config.json'),'checkpoint_sha256':result.get('checkpoint_sha256'),
                'events_path':str((d/'events.jsonl').relative_to(PHASE))}
            episodes.append(ep)
        candidates=[]
        for seed in range(3001,3011):
            rows=trajectories[condition+'-'+str(seed)]
            for start in range(len(rows)-59):
                segment=rows[start:start+60]
                if sum(r['b'] for r in segment)==0:
                    candidates.append((sum(r['affect_impulse'] for r in segment),-seed,-start,seed,start,segment))
        chosen=max(candidates,key=lambda v:v[:3])
        for kind,seed,start,segment in [('high_q_zero_progress',chosen[3],chosen[4],chosen[5]),('ordinary_fixed',3001,30,trajectories[condition+'-3001'][30:90])]:
            segments.append({'condition':condition,'selection':kind,'seed':seed,'start_decision':start+1,'end_decision':start+len(segment),
                'q_impulse_sum':sum(r['affect_impulse'] for r in segment),'delivered_reward_sum':sum(r['delivered_reward'] for r in segment),
                'task_events':sum(r['b'] for r in segment),'score_start':segment[0]['raw_score_signal'],'score_end':segment[-1]['raw_score_signal'],
                'valid_pairs':sum(r['fresh'] for r in segment),'interpretation':'High q alone is preference-model support, not task completion or measured human affect.'})
    assert len(pids)==30 and len(states)==1
    baselines=[]
    for c in ['TASK','MAX','RANDOM']:
        selected=[r for r in episodes if r['condition']==c];scores=[r['raw_score'] for r in selected]
        baselines.append({'condition':c,'episodes':len(selected),'training_decisions':selected[0]['training_decisions'],
            'raw_score_mean':float(np.mean(scores)),'raw_score_median':float(np.median(scores)),'raw_score_min':min(scores),'raw_score_max':max(scores),
            'success_completion_count':None,'completion_note':'Success/failure labels not established by frozen telemetry; do not equate timeout or score checkpoint with completion.',
            'natural_terminal_count':sum(r['terminated'] for r in selected),'zero_progress_episodes':sum(r['zero_progress'] for r in selected),
            'q_valid_pair_mean':float(np.average([r['q_mean'] for r in selected],weights=[r['valid_pairs'] for r in selected])),
            'valid_pairs':sum(r['valid_pairs'] for r in selected),'eligible_pairs':sum(r['eligible_pairs'] for r in selected),
            'pair_coverage':sum(r['valid_pairs'] for r in selected)/sum(r['eligible_pairs'] for r in selected),
            'missing_eligible_pairs':sum(r['missing_eligible_pairs'] for r in selected),'early_endings':sum(r['decisions']<600 for r in selected),
            'wrapper_timeouts':sum(r['end_reason']=='wrapper_decision_limit' for r in selected),'evaluation_worker_seconds':sum(r['worker_wall_seconds'] for r in selected),
            'distinct_recorded_trajectories':len(set(r['trajectory_sha256'] for r in selected))})
    csv_out(ROOT/'baseline_results.csv',baselines);csv_out(ROOT/'evaluation_episodes.csv',episodes);csv_out(ROOT/'valid_pairs.csv',pair_rows);csv_out(ROOT/'segment_review.csv',segments)
    write_json(ROOT/'plot_data.json',trajectories);write_json(ROOT/'random_runtime_build_inventories.json',random_builds)
    write_json(ROOT/'verification.json',{'status':'PASS','evaluation_episodes':30,'evaluation_decisions':all_decisions,'valid_pairs':len(pair_rows),
        'maximum_abs_errors':errors,'frozen_estimator_state_sha256':next(iter(states)),'fresh_worker_processes':len(pids),'random_action_sequences_exactly_replayed':True,
        'limits':['TASK and MAX each have one trained policy; repeated episodes do not replicate training.','Requested Unity seed diversity unproven; RANDOM diversity also changes action RNG.','Success completion labels unavailable; explicit NA, natural endings and timeouts separately recorded.']})

    return baselines
print('Independent raw-event analysis definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Independent raw-event analysis definitions/execution completed.
